In [4]:
!pip install opencv-python

In [1]:
# mount drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 112.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 101.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 36.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lit: filename=lit-15.0.7-py3-none-any.whl size=89991 sha256=677b7dd172cfdca056c3da94177d296cf69a1dc52ef1c65e4d42ec14b242833c
  Stored in directory: /root/.cache/pip/wheels/fc/5d/45/34fe9945d5e45e261134e72284395be36c2d4828af38e2b0fe
Successfully built lit
  Attempting uninstall: triton
    Found existing installation: triton 3.1.0
    Uninstalling triton-3.1.0:
      Successfully uninstalled triton-3.1.0
  Attempting uninstall: torch
    Found existing installation: torch 2.

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large
from PIL import Image
import numpy as np
import sys
from tqdm import tqdm
import time
import matplotlib.pyplot as plt
import cv2


In [6]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = sorted(os.listdir(image_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.images[idx].replace('.jpg', '_mask.png'))

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        mask = (mask > 0).float()  # Convert to binary
        return image, mask


In [7]:
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])

train_image_path = '/content/drive/MyDrive/Projects/PR/Ground_segmentation/dataset/split_dataset/train/images'
train_mask_path = '/content/drive/MyDrive/Projects/PR/Ground_segmentation/dataset/split_dataset/train/masks'

val_image_path = '/content/drive/MyDrive/Projects/PR/Ground_segmentation/dataset/split_dataset/val/images'
val_mask_path = '/content/drive/MyDrive/Projects/PR/Ground_segmentation/dataset/split_dataset/val/masks'


test_image_path = '/content/drive/MyDrive/Projects/PR/Ground_segmentation/dataset/split_dataset/test/images'
test_mask_path = '/content/drive/MyDrive/Projects/PR/Ground_segmentation/dataset/split_dataset/test/masks'


print(len(os.listdir(train_image_path)))
print(len(os.listdir(train_mask_path)))

print(len(os.listdir(val_image_path)))
print(len(os.listdir(val_mask_path)))

print(len(os.listdir(test_image_path)))
print(len(os.listdir(test_mask_path)))

train_dataset = SegmentationDataset(train_image_path, train_mask_path, transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

val_dataset = SegmentationDataset(val_image_path, val_mask_path, transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)


719
719
154
154
155
155


In [8]:
model = deeplabv3_mobilenet_v3_large(pretrained=True)
model.classifier[4] = nn.Conv2d(256, 1, kernel_size=1)
model = model.cuda()


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/deeplabv3_mobilenet_v3_large-fc3c493d.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_mobilenet_v3_large-fc3c493d.pth
100%|██████████| 42.3M/42.3M [00:00<00:00, 288MB/s]


In [9]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [14]:
def compute_iou(preds, targets, threshold=0.5):
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = ((preds + targets) >= 1).float().sum(dim=(1, 2, 3))
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.mean().item()


In [15]:
os.chdir('/content/drive/MyDrive/Projects/PR/Ground_segmentation/')
os.getcwd()

'/content/drive/MyDrive/Projects/PR/Ground_segmentation'

In [16]:
def evaluate(model, val_loader):
    model.eval()
    total_val_loss = 0
    total_val_iou = 0

    val_loop = tqdm(val_loader, desc="Validating", unit="batch")

    with torch.no_grad():
        for images, masks in val_loop:
            images, masks = images.cuda(), masks.cuda()
            outputs = model(images)['out']
            val_loss = criterion(outputs, masks)
            val_iou = compute_iou(outputs, masks)

            total_val_loss += val_loss.item()
            total_val_iou += val_iou

            val_loop.set_postfix({
                "ValLoss": f"{val_loss.item():.4f}",
                "ValIoU": f"{val_iou:.4f}"
            })

    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_iou = total_val_iou / len(val_loader)

    model.train()
    return avg_val_loss, avg_val_iou

In [17]:
len(val_loader)

154

In [19]:
best_iou = 0.0
num_epochs = 70

model.train()

for epoch in range(num_epochs):
    total_train_loss = 0
    total_train_iou = 0
    epoch_start = time.time()

    print(f"\n Epoch {epoch+1}/{num_epochs}")
    loop = tqdm(train_loader, desc="Training", unit="batch")

    for images, masks in loop:
        images, masks = images.cuda(), masks.cuda()
        outputs = model(images)['out']
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_iou = compute_iou(outputs, masks)
        total_train_loss += loss.item()
        total_train_iou += batch_iou

        loop.set_postfix({
            "TrainLoss": f"{loss.item():.4f}",
            "TrainIoU": f"{batch_iou:.4f}"
        })

    avg_train_loss = total_train_loss / len(train_loader)
    avg_train_iou = total_train_iou / len(train_loader)

    # Run validation
    avg_val_loss, avg_val_iou = evaluate(model, val_loader)

    epoch_time = time.time() - epoch_start
    print(f"\n Epoch {epoch+1} Summary:")
    print(f" Train   — Loss: {avg_train_loss:.4f} | IoU: {avg_train_iou:.4f}")
    print(f" Val     — Loss: {avg_val_loss:.4f} | IoU: {avg_val_iou:.4f}")
    print(f"⏱ Time: {epoch_time:.2f} sec")

    # Save best model based on validation IoU
    if avg_val_iou > best_iou:
        best_iou = avg_val_iou
        model_filename = f"/content/drive/MyDrive/Projects/PR/Ground_segmentation/MODELS/Best_model_epoch_{epoch+1:02d}_iou_{avg_val_iou:.4f}_loss_{avg_val_loss:.4f}.pth"
        torch.save(model.state_dict(), model_filename)
        print(f" Best model saved as: {model_filename}")


 Epoch 1/70


Validating: 100%|██████████| 154/154 [00:02<00:00, 68.71batch/s, ValLoss=0.0278, ValIoU=0.9700]



 Epoch 1 Summary:
 Train   — Loss: 0.0496 | IoU: 0.9544
 Val     — Loss: 0.0419 | IoU: 0.9593
⏱ Time: 22.47 sec
 Best model saved as: /content/drive/MyDrive/Projects/PR/Ground_segmentation/MODELS/Best_model_epoch_01_iou_0.9593_loss_0.0419.pth

 Epoch 2/70


Validating: 100%|██████████| 154/154 [00:02<00:00, 65.27batch/s, ValLoss=0.0214, ValIoU=0.9757]



 Epoch 2 Summary:
 Train   — Loss: 0.0388 | IoU: 0.9610
 Val     — Loss: 0.0362 | IoU: 0.9632
⏱ Time: 22.63 sec
 Best model saved as: /content/drive/MyDrive/Projects/PR/Ground_segmentation/MODELS/Best_model_epoch_02_iou_0.9632_loss_0.0362.pth

 Epoch 3/70


...


 Epoch 68 Summary:
 Train   — Loss: 0.0127 | IoU: 0.9842
 Val     — Loss: 0.0328 | IoU: 0.9714
⏱ Time: 22.31 sec

 Epoch 69/70


Validating: 100%|██████████| 154/154 [00:02<00:00, 69.44batch/s, ValLoss=0.0187, ValIoU=0.9827]



 Epoch 69 Summary:
 Train   — Loss: 0.0126 | IoU: 0.9842
 Val     — Loss: 0.0311 | IoU: 0.9711
⏱ Time: 22.35 sec

 Epoch 70/70


Validating: 100%|██████████| 154/154 [00:02<00:00, 69.36batch/s, ValLoss=0.0198, ValIoU=0.9812]


 Epoch 70 Summary:
 Train   — Loss: 0.0126 | IoU: 0.9843
 Val     — Loss: 0.0329 | IoU: 0.9710
⏱ Time: 22.74 sec
